# 🚀 AI Stock Analyzer

This notebook provides:
- **10-K Filing Analysis** — Fetches real SEC EDGAR filings, fully formatted
- **Portfolio Holdings** — Load your real holdings from `holdings.csv`
- **ML Price Forecasting** — Prophet + Random Forest on live data
- **GPU Detection** — Shows whether computations use GPU or CPU

---

In [ ]:
# ── 0. Widen notebook output to full screen ──────────────────────────────────
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; } "
             ".output_scroll { height: auto !important; max-height: none !important; } "
             "div.cell.selected { border-left-width: 3px; } "
             ".jp-Cell { width: 100% !important; }</style>"))

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import sys
import json
import re
import time
import warnings
import textwrap
from datetime import datetime, timedelta
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML, Markdown
from bs4 import BeautifulSoup
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import ta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✅ Imports complete')

In [ ]:
# ── 2. GPU Detection ──────────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
        device   = torch.device('cuda')
        display(HTML(
            f"<div style='background:#1a1a2e;color:#00ff88;padding:16px;border-radius:8px;"
            f"font-size:16px;font-family:monospace;border:2px solid #00ff88;'>"
            f"🟢 <b>GPU ACTIVE</b> — {gpu_name} ({gpu_mem:.1f} GB VRAM)<br>"
            f"All heavy computations (LSTM, tensor ops) will use the <b>GPU</b>."
            f"</div>"
        ))
    else:
        device = torch.device('cpu')
        display(HTML(
            "<div style='background:#2d1b00;color:#ffaa00;padding:16px;border-radius:8px;"
            "font-size:16px;font-family:monospace;border:2px solid #ffaa00;'>"
            "🟡 <b>CPU MODE</b> — No CUDA GPU detected.<br>"
            "Models will run on the <b>CPU</b>. Install CUDA + PyTorch-GPU to enable GPU."
            "</div>"
        ))
except ImportError:
    device = None
    display(HTML(
        "<div style='background:#2d0000;color:#ff6666;padding:16px;border-radius:8px;"
        "font-size:16px;font-family:monospace;border:2px solid #ff6666;'>"
        "⚠️ <b>PyTorch not installed</b> — run <code>pip install torch</code> to enable GPU support."
        "</div>"
    ))

print(f'Device in use: {device}')

In [ ]:
# ── 3. Configuration — edit tickers / date range here ─────────────────────────
HOLDINGS_FILE  = Path('holdings.csv')   # path to your holdings CSV
DEFAULT_TICKER = 'MU'                   # fallback ticker for 10-K demo
LOOKBACK_YEARS = 3                      # years of price history to fetch
FORECAST_DAYS  = 90                     # days ahead to forecast

END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * LOOKBACK_YEARS)

print(f'Date range  : {START_DATE.date()} → {END_DATE.date()}')
print(f'Holdings    : {HOLDINGS_FILE}')
print(f'Default tick: {DEFAULT_TICKER}')

---
## 📂 Section 1 — Portfolio Holdings (Real Data)

In [ ]:
# ── 4. Load holdings ──────────────────────────────────────────────────────────
if HOLDINGS_FILE.exists():
    holdings = pd.read_csv(HOLDINGS_FILE)
    print(f'Loaded {len(holdings)} holdings from {HOLDINGS_FILE}')
else:
    raise FileNotFoundError(
        f'{HOLDINGS_FILE} not found. '
        f'Place a CSV with columns: ticker, shares, avg_cost, sector'
    )

# Fetch live prices for every holding
print('Fetching live prices …')
live_rows = []
for _, row in holdings.iterrows():
    try:
        info  = yf.Ticker(row['ticker']).fast_info
        price = info.last_price
    except Exception:
        price = np.nan
    live_rows.append(price)

holdings['current_price'] = live_rows
holdings['market_value']  = holdings['shares'] * holdings['current_price']
holdings['cost_basis']    = holdings['shares'] * holdings['avg_cost']
holdings['gain_loss']     = holdings['market_value'] - holdings['cost_basis']
holdings['gain_loss_pct'] = np.where(
    holdings['cost_basis'] != 0,
    (holdings['gain_loss'] / holdings['cost_basis']) * 100,
    np.nan
)

total_value    = holdings['market_value'].sum()
total_cost     = holdings['cost_basis'].sum()
total_gain     = holdings['gain_loss'].sum()
total_gain_pct = (total_gain / total_cost) * 100 if total_cost != 0 else 0.0

display(HTML(
    f"<div style='background:#0d1117;color:#c9d1d9;padding:20px;border-radius:10px;"
    f"font-size:15px;border:1px solid #30363d;'>"
    f"<b>📊 Portfolio Summary</b><br><br>"
    f"Total Market Value: <b style='color:#58a6ff'>${total_value:,.2f}</b><br>"
    f"Total Cost Basis : <b>${total_cost:,.2f}</b><br>"
    f"Total Gain/Loss  : <b style='color:{('#3fb950' if total_gain>=0 else '#f85149')}'>"
    f"${total_gain:,.2f} ({total_gain_pct:+.2f}%)</b>"
    f"</div>"
))

display(holdings.sort_values('market_value', ascending=False)
                .reset_index(drop=True))

In [ ]:
# ── 5. Portfolio allocation charts ───────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=['Allocation by Market Value', 'Gain / Loss by Position']
)

fig.add_trace(
    go.Pie(labels=holdings['ticker'], values=holdings['market_value'],
           hole=0.4, textinfo='label+percent'),
    row=1, col=1
)
colors = ['#3fb950' if g >= 0 else '#f85149' for g in holdings['gain_loss']]
fig.add_trace(
    go.Bar(x=holdings['ticker'], y=holdings['gain_loss'],
           marker_color=colors, text=holdings['gain_loss_pct'].apply(lambda x: f'{x:+.1f}%'),
           textposition='outside'),
    row=1, col=2
)
fig.update_layout(
    height=500, width=1400,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    title_text='Portfolio Overview — Real Holdings',
    title_font_size=20
)
fig.show()

---
## 📈 Section 2 — Real Price History & Technical Analysis

In [ ]:
# ── 6. Fetch historical OHLCV for all holdings ────────────────────────────────
tickers_list = holdings['ticker'].tolist()
raw = yf.download(
    tickers_list,
    start=START_DATE.strftime('%Y-%m-%d'),
    end=END_DATE.strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=True
)

# Normalise to single-ticker or multi-ticker structure
if isinstance(raw.columns, pd.MultiIndex):
    close_prices = raw['Close']
else:
    close_prices = raw[['Close']].rename(columns={'Close': tickers_list[0]})

print(f'Downloaded {len(close_prices)} trading days for {close_prices.shape[1]} tickers')
display(close_prices.tail())

In [ ]:
# ── 7. Normalised price performance chart ─────────────────────────────────────
# Use first valid price for each ticker to avoid divide-by-zero on newly-listed tickers
first_valid = close_prices.apply(lambda col: col.dropna().iloc[0] if col.dropna().size > 0 else np.nan)
norm = (close_prices / first_valid) * 100

fig = go.Figure()
for col in norm.columns:
    fig.add_trace(go.Scatter(x=norm.index, y=norm[col], mode='lines', name=col))

fig.update_layout(
    title='Normalised Price Performance (base=100)',
    xaxis_title='Date', yaxis_title='Index (100 = start)',
    height=550, width=1400,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [ ]:
# ── 8. Deep-dive on a single ticker with technical indicators ─────────────────
FOCUS_TICKER = DEFAULT_TICKER  # change as desired, e.g. 'NVDA'

df_focus = yf.download(
    FOCUS_TICKER,
    start=START_DATE.strftime('%Y-%m-%d'),
    end=END_DATE.strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=False
).copy()

df_focus.columns = [c[0] if isinstance(c, tuple) else c for c in df_focus.columns]

# Technical indicators via 'ta' library
df_focus['SMA_20']  = ta.trend.sma_indicator(df_focus['Close'], window=20)
df_focus['SMA_50']  = ta.trend.sma_indicator(df_focus['Close'], window=50)
df_focus['EMA_12']  = ta.trend.ema_indicator(df_focus['Close'], window=12)
df_focus['RSI']     = ta.momentum.rsi(df_focus['Close'], window=14)
df_focus['MACD']    = ta.trend.macd(df_focus['Close'])
df_focus['MACD_sig']= ta.trend.macd_signal(df_focus['Close'])
df_focus['BB_upper']= ta.volatility.bollinger_hband(df_focus['Close'])
df_focus['BB_lower']= ta.volatility.bollinger_lband(df_focus['Close'])
df_focus['ATR']     = ta.volatility.average_true_range(
                          df_focus['High'], df_focus['Low'], df_focus['Close'])
df_focus.dropna(inplace=True)

print(f'{FOCUS_TICKER}: {len(df_focus)} trading days with indicators')

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.25, 0.20],
    subplot_titles=[f'{FOCUS_TICKER} Price + BB + SMA', 'RSI (14)', 'MACD']
)

# Candlestick
fig.add_trace(go.Candlestick(
    x=df_focus.index,
    open=df_focus['Open'], high=df_focus['High'],
    low=df_focus['Low'],   close=df_focus['Close'],
    name='OHLC', increasing_line_color='#3fb950', decreasing_line_color='#f85149'
), row=1, col=1)

for col, color, dash in [('SMA_20','#58a6ff','solid'),
                          ('SMA_50','#ff9f1c','dash'),
                          ('BB_upper','#8b949e','dot'),
                          ('BB_lower','#8b949e','dot')]:
    fig.add_trace(go.Scatter(
        x=df_focus.index, y=df_focus[col],
        mode='lines', name=col,
        line=dict(color=color, dash=dash, width=1.5)
    ), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(
    x=df_focus.index, y=df_focus['RSI'],
    mode='lines', name='RSI', line=dict(color='#d2a8ff')
), row=2, col=1)
for level, color in [(70,'#f85149'),(30,'#3fb950')]:
    fig.add_hline(y=level, line_dash='dash', line_color=color, row=2, col=1)

# MACD
fig.add_trace(go.Scatter(
    x=df_focus.index, y=df_focus['MACD'],
    mode='lines', name='MACD', line=dict(color='#58a6ff')
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df_focus.index, y=df_focus['MACD_sig'],
    mode='lines', name='Signal', line=dict(color='#ff9f1c')
), row=3, col=1)

fig.update_layout(
    height=850, width=1400,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    xaxis_rangeslider_visible=False,
    showlegend=True
)
fig.show()

---
## 📄 Section 3 — 10-K Filing Analysis (SEC EDGAR)

In [ ]:
# ── 9. SEC EDGAR helpers ──────────────────────────────────────────────────────
SEC_EDGAR_RATE_LIMIT_DELAY = 0.15  # seconds between EDGAR requests (≤ 10 req/s)

EDGAR_HEADERS = {
    'User-Agent': 'AI-Stock-Analyzer research@example.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'data.sec.gov'
}

_TICKER_MAP: dict = {}  # module-level cache for company_tickers.json


def get_cik(ticker: str) -> str:
    """Return zero-padded 10-digit CIK for a ticker (result cached in _TICKER_MAP)."""
    global _TICKER_MAP
    if not _TICKER_MAP:
        mapping_url = 'https://www.sec.gov/files/company_tickers.json'
        r = requests.get(mapping_url,
                         headers={'User-Agent': 'AI-Stock-Analyzer research@example.com'},
                         timeout=30)
        r.raise_for_status()
        _TICKER_MAP = r.json()
    ticker_upper = ticker.upper()
    for entry in _TICKER_MAP.values():
        if entry['ticker'].upper() == ticker_upper:
            return str(entry['cik_str']).zfill(10)
    raise ValueError(f'CIK not found for ticker: {ticker}')


def get_10k_filings(cik: str, limit: int = 3) -> list:
    """Return a list of dicts with accession info for recent 10-K filings."""
    url = f'https://data.sec.gov/submissions/CIK{cik}.json'
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=30)
    r.raise_for_status()
    sub = r.json()
    filings = sub['filings']['recent']
    results = []
    for i, form in enumerate(filings['form']):
        if form == '10-K':
            acc = filings['accessionNumber'][i].replace('-', '')
            results.append({
                'date':       filings['filingDate'][i],
                'accession':  filings['accessionNumber'][i],
                'acc_clean':  acc,
                'primary':    filings['primaryDocument'][i]
            })
            if len(results) >= limit:
                break
    return results


def fetch_10k_text(cik: str, accession_clean: str, primary_doc: str) -> str:
    """Download and clean a 10-K filing from EDGAR."""
    url  = f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_clean}/{primary_doc}'
    r = requests.get(url, headers={'User-Agent': 'AI-Stock-Analyzer research@example.com'}, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.content, 'lxml')
    # Remove scripts, styles, and invisible elements
    for tag in soup(['script', 'style', 'head', 'meta', 'link']):
        tag.decompose()
    text = soup.get_text(separator='\n')
    # Collapse excessive blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def extract_sections(text: str) -> dict:
    """Extract key 10-K sections by their item headings."""
    section_patterns = [
        ('Business',          r'Item\s*1[^A-Z].*?Business'),
        ('Risk Factors',      r'Item\s*1A'),
        ('MD&A',              r'Item\s*7[^A-Z]'),
        ('Financial Statements', r'Item\s*8[^A-Z]'),
        ('Quantitative Risk', r'Item\s*7A'),
    ]
    lines   = text.split('\n')
    indices = {}
    for name, pattern in section_patterns:
        for i, line in enumerate(lines):
            if re.search(pattern, line, re.IGNORECASE):
                indices[name] = i
                break
    # Extract text between consecutive section markers
    sorted_items = sorted(indices.items(), key=lambda x: x[1])
    sections     = {}
    for idx, (name, start) in enumerate(sorted_items):
        end = sorted_items[idx + 1][1] if idx + 1 < len(sorted_items) else start + 600
        sections[name] = '\n'.join(lines[start:min(start + 400, end)])
    return sections


print('✅ SEC EDGAR helpers ready')

In [ ]:
# ── 10. Fetch & display 10-K for FOCUS_TICKER ─────────────────────────────────
print(f'Looking up CIK for {FOCUS_TICKER} …')
cik_padded = get_cik(FOCUS_TICKER)
print(f'CIK: {cik_padded}')

time.sleep(SEC_EDGAR_RATE_LIMIT_DELAY)  # respect SEC EDGAR rate limit (max ~10 req/s)
filings_meta = get_10k_filings(cik_padded, limit=3)
sections = {}   # initialised here so Cell 11 never gets a NameError
latest   = None
if not filings_meta:
    print(f'No 10-K filings found for {FOCUS_TICKER}')
else:
    # Display filing index
    df_filings = pd.DataFrame(filings_meta)[['date', 'accession', 'primary']]
    display(HTML('<h3 style="color:#58a6ff">Available 10-K Filings</h3>'))
    display(df_filings)

    # Fetch the most recent 10-K
    latest = filings_meta[0]
    print(f"\nFetching most recent 10-K ({latest['date']}) …")
    try:
        filing_text = fetch_10k_text(cik_padded, latest['acc_clean'], latest['primary'])
        sections    = extract_sections(filing_text)
        print(f'Extracted {len(sections)} sections: {list(sections.keys())}')
        print(f'Total text length: {len(filing_text):,} characters')
    except Exception as e:
        print(f'Error fetching 10-K text: {e}')
        filing_text = ''
        sections    = {}

In [ ]:
# ── 11. Render 10-K sections in a clean full-width format ─────────────────────
SECTION_COLORS = {
    'Business':             '#58a6ff',
    'Risk Factors':         '#f85149',
    'MD&A':                 '#3fb950',
    'Financial Statements': '#ff9f1c',
    'Quantitative Risk':    '#d2a8ff',
}

def render_section(title: str, text: str, color: str = '#58a6ff',
                   max_chars: int = 6000) -> None:
    """Display a 10-K section in a styled, full-width panel."""
    snippet = text[:max_chars] + (' …[truncated]' if len(text) > max_chars else '')
    # Escape HTML entities
    snippet_html = snippet.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    snippet_html = snippet_html.replace('\n', '<br>')
    html = (
        f"<div style='width:100%;box-sizing:border-box;background:#0d1117;"
        f"border-left:5px solid {color};border-radius:4px;"
        f"padding:20px 24px;margin:12px 0;font-family:Consolas,monospace;"
        f"font-size:13px;color:#c9d1d9;line-height:1.7;'>"
        f"<div style='font-size:18px;font-weight:bold;color:{color};"
        f"margin-bottom:12px;border-bottom:1px solid {color};padding-bottom:8px;'>"
        f"📋 {title}</div>"
        f"{snippet_html}"
        f"</div>"
    )
    display(HTML(html))


if sections:
    display(HTML(
        f"<h2 style='color:#c9d1d9;border-bottom:2px solid #30363d;padding-bottom:8px;'>"
        f"📄 10-K Analysis — {FOCUS_TICKER} &nbsp;"
        f"<span style='font-size:14px;color:#8b949e;'>(Filed: {latest['date']})</span></h2>"
    ))
    for sec_name, sec_text in sections.items():
        color = SECTION_COLORS.get(sec_name, '#58a6ff')
        render_section(sec_name, sec_text, color)
else:
    display(HTML("<p style='color:#f85149'>⚠️ No sections extracted. Check network access to SEC EDGAR.</p>"))

In [ ]:
# ── 12. Key financial metrics from EDGAR XBRL ─────────────────────────────────
def get_xbrl_facts(cik: str, concept: str, unit: str = 'USD') -> pd.DataFrame:
    """Fetch a single XBRL concept time-series from EDGAR company-facts API."""
    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'
    try:
        r = requests.get(url, headers=EDGAR_HEADERS, timeout=30)
        r.raise_for_status()
        facts = r.json()
        entries = facts['facts']['us-gaap'][concept]['units'][unit]
        df = pd.DataFrame(entries)
        df['end'] = pd.to_datetime(df['end'])
        # Keep annual 10-K values only
        df = df[df['form'] == '10-K'].copy() if 'form' in df.columns else df
        df = df.sort_values('end').drop_duplicates('end', keep='last')
        return df[['end', 'val']].rename(columns={'end': 'Date', 'val': concept})
    except Exception:
        return pd.DataFrame()


metrics = {
    'Revenues':              'Revenues',
    'Net Income':            'NetIncomeLoss',
    'Operating Income':      'OperatingIncomeLoss',
    'R&D Expense':           'ResearchAndDevelopmentExpense',
    'Total Assets':          'Assets',
    'Long Term Debt':        'LongTermDebt',
}

financial_dfs = {}
print('Fetching XBRL financial data …')
for label, concept in metrics.items():
    df_m = get_xbrl_facts(cik_padded, concept)
    if not df_m.empty:
        df_m = df_m.rename(columns={concept: label})
        financial_dfs[label] = df_m
        print(f'  ✅ {label}: {len(df_m)} annual data points')
    else:
        print(f'  ⚠️  {label}: no data')

print('Done.')

In [ ]:
# ── 13. Financial metrics visualisation ───────────────────────────────────────
if financial_dfs:
    n_metrics = len(financial_dfs)
    cols = 2
    rows = (n_metrics + 1) // cols
    fig = make_subplots(
        rows=rows, cols=cols,
        subplot_titles=list(financial_dfs.keys())
    )
    palette = ['#58a6ff','#3fb950','#ff9f1c','#f85149','#d2a8ff','#79c0ff']
    for idx, (label, df_m) in enumerate(financial_dfs.items()):
        r = idx // cols + 1
        c = idx %  cols + 1
        fig.add_trace(
            go.Bar(x=df_m['Date'].dt.year, y=df_m[label] / 1e9,
                   name=label, marker_color=palette[idx % len(palette)]),
            row=r, col=c
        )
    fig.update_layout(
        title_text=f'{FOCUS_TICKER} — Annual Financial Metrics (Billions USD)',
        height=400 * rows, width=1400,
        paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
        font=dict(color='#c9d1d9'),
        showlegend=False
    )
    fig.show()
else:
    print('No financial data available to plot')

---
## 🤖 Section 4 — ML Price Forecasting on Real Data

In [ ]:
# ── 14. Feature engineering for ML ───────────────────────────────────────────
df_ml = df_focus[['Close','Volume','SMA_20','SMA_50','RSI','MACD','ATR']].copy()
df_ml['Return_1d']  = df_ml['Close'].pct_change(1)
df_ml['Return_5d']  = df_ml['Close'].pct_change(5)
df_ml['Return_20d'] = df_ml['Close'].pct_change(20)
df_ml['Volatility'] = df_ml['Return_1d'].rolling(20).std()
df_ml['Volume_MA']  = df_ml['Volume'].rolling(20).mean()
df_ml['Target']     = df_ml['Close'].shift(-5)  # predict 5-day forward price
df_ml.dropna(inplace=True)

FEATURES = ['Close','SMA_20','SMA_50','RSI','MACD','ATR',
            'Return_1d','Return_5d','Return_20d','Volatility','Volume_MA']
X = df_ml[FEATURES].values
y = df_ml['Target'].values

# Train/test split (no shuffle — time series)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_test      = df_ml.index[split:]

scaler   = MinMaxScaler()
X_train_ = scaler.fit_transform(X_train)
X_test_  = scaler.transform(X_test)

print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')

In [ ]:
# ── 15. Random Forest ─────────────────────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_, y_train)
rf_pred = rf.predict(X_test_)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)

display(HTML(
    f"<div style='background:#0d1117;color:#c9d1d9;padding:16px;border-radius:8px;"
    f"border:1px solid #3fb950;'>"
    f"<b style='color:#3fb950'>🌲 Random Forest — 5-day forward price forecast</b><br><br>"
    f"RMSE : <b>${rf_rmse:.2f}</b><br>"
    f"MAE  : <b>${rf_mae:.2f}</b><br>"
    f"R²   : <b>{rf_r2:.4f}</b>"
    f"</div>"
))

# Feature importances
imp_df = (pd.DataFrame({'feature': FEATURES, 'importance': rf.feature_importances_})
          .sort_values('importance', ascending=True))

fig = go.Figure(go.Bar(
    x=imp_df['importance'], y=imp_df['feature'],
    orientation='h', marker_color='#3fb950'
))
fig.update_layout(
    title='Random Forest — Feature Importances',
    height=400, width=900,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9')
)
fig.show()

In [ ]:
# ── 16. Gradient Boosting ─────────────────────────────────────────────────────
gb = GradientBoostingRegressor(n_estimators=300, max_depth=5,
                                learning_rate=0.05, random_state=42)
gb.fit(X_train_, y_train)
gb_pred = gb.predict(X_test_)

gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_mae  = mean_absolute_error(y_test, gb_pred)
gb_r2   = r2_score(y_test, gb_pred)

display(HTML(
    f"<div style='background:#0d1117;color:#c9d1d9;padding:16px;border-radius:8px;"
    f"border:1px solid #ff9f1c;'>"
    f"<b style='color:#ff9f1c'>🚀 Gradient Boosting — 5-day forward price forecast</b><br><br>"
    f"RMSE : <b>${gb_rmse:.2f}</b><br>"
    f"MAE  : <b>${gb_mae:.2f}</b><br>"
    f"R²   : <b>{gb_r2:.4f}</b>"
    f"</div>"
))

In [ ]:
# ── 17. Prophet forecast (requires prophet package) ───────────────────────────
try:
    from prophet import Prophet

    prophet_df = df_focus[['Close']].reset_index()
    prophet_df.columns = ['ds', 'y']
    prophet_df['ds'] = pd.to_datetime(prophet_df['ds']).dt.tz_localize(None)

    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05
    )
    m.fit(prophet_df)

    future   = m.make_future_dataframe(periods=FORECAST_DAYS)
    forecast = m.predict(future)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=prophet_df['ds'], y=prophet_df['y'],
        mode='lines', name='Actual', line=dict(color='#c9d1d9')
    ))
    fig.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat'],
        mode='lines', name='Forecast', line=dict(color='#58a6ff', dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
        y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
        fill='toself', fillcolor='rgba(88,166,255,0.15)',
        line=dict(color='rgba(255,255,255,0)'),
        name='Confidence Band'
    ))
    fig.update_layout(
        title=f'{FOCUS_TICKER} — Prophet {FORECAST_DAYS}-day Forecast',
        xaxis_title='Date', yaxis_title='Price (USD)',
        height=500, width=1400,
        paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
        font=dict(color='#c9d1d9')
    )
    fig.show()

    last_forecast = forecast.iloc[-1]
    display(HTML(
        f"<div style='background:#0d1117;color:#c9d1d9;padding:16px;border-radius:8px;"
        f"border:1px solid #d2a8ff;'>"
        f"<b style='color:#d2a8ff'>🔮 Prophet {FORECAST_DAYS}-day Forecast</b><br><br>"
        f"Forecast Date   : <b>{last_forecast['ds'].date()}</b><br>"
        f"Predicted Price : <b>${last_forecast['yhat']:.2f}</b><br>"
        f"Upper Bound     : <b>${last_forecast['yhat_upper']:.2f}</b><br>"
        f"Lower Bound     : <b>${last_forecast['yhat_lower']:.2f}</b>"
        f"</div>"
    ))

except ImportError:
    display(HTML(
        "<div style='background:#2d1b00;color:#ffaa00;padding:12px;border-radius:8px;"
        "border:1px solid #ffaa00;'>"
        "⚠️ Prophet not installed — run <code>pip install prophet</code> to enable."
        "</div>"
    ))

In [ ]:
# ── 18. ML model comparison chart ─────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=dates_test, y=y_test, mode='lines', name='Actual',
    line=dict(color='#c9d1d9', width=2)
))
fig.add_trace(go.Scatter(
    x=dates_test, y=rf_pred, mode='lines', name='Random Forest',
    line=dict(color='#3fb950', width=1.5, dash='dot')
))
fig.add_trace(go.Scatter(
    x=dates_test, y=gb_pred, mode='lines', name='Gradient Boosting',
    line=dict(color='#ff9f1c', width=1.5, dash='dash')
))
fig.update_layout(
    title=f'{FOCUS_TICKER} — Model Predictions vs Actual (Test Set)',
    xaxis_title='Date', yaxis_title='Price (USD)',
    height=500, width=1400,
    paper_bgcolor='#0d1117', plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

# Summary table
summary_df = pd.DataFrame([
    {'Model': 'Random Forest',     'RMSE': rf_rmse, 'MAE': rf_mae, 'R²': rf_r2},
    {'Model': 'Gradient Boosting', 'RMSE': gb_rmse, 'MAE': gb_mae, 'R²': gb_r2},
])
display(HTML('<h3 style="color:#58a6ff">Model Performance Summary</h3>'))
display(summary_df.set_index('Model').style
        .format({'RMSE': '${:.2f}', 'MAE': '${:.2f}', 'R²': '{:.4f}'})
        .background_gradient(cmap='Greens', subset=['R²'])
        .set_table_styles([{'selector': 'th', 'props': [('color', '#58a6ff')]}]))

---
## 🏦 Section 5 — Run Models on All Holdings

In [ ]:
# ── 19. Quick forecast for every holding ──────────────────────────────────────
def quick_forecast(ticker: str, periods: int = 5) -> dict:
    """Return RF 5-day forward price prediction for a ticker."""
    try:
        df = yf.download(ticker, period='2y', auto_adjust=True, progress=False)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        if len(df) < 60:
            return {'ticker': ticker, 'error': 'Insufficient data'}

        df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
        df['SMA_50'] = ta.trend.sma_indicator(df['Close'], window=50)
        df['RSI']    = ta.momentum.rsi(df['Close'], window=14)
        df['MACD']   = ta.trend.macd(df['Close'])
        df['ATR']    = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'])
        df['R1']     = df['Close'].pct_change(1)
        df['R5']     = df['Close'].pct_change(5)
        df['Vol']    = df['R1'].rolling(20).std()
        df['Target'] = df['Close'].shift(-periods)
        df.dropna(inplace=True)

        feats = ['Close','SMA_20','SMA_50','RSI','MACD','ATR','R1','R5','Vol']
        X = df[feats].values
        y = df['Target'].values
        split  = int(len(X) * 0.8)
        sc     = MinMaxScaler().fit(X[:split])
        rf_q   = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_q.fit(sc.transform(X[:split]), y[:split])

        current_price  = float(df['Close'].iloc[-1])
        last_features  = sc.transform(X[-1].reshape(1, -1))
        predicted_price = float(rf_q.predict(last_features)[0])
        expected_return = (predicted_price - current_price) / current_price * 100

        return {
            'ticker':          ticker,
            'current_price':   current_price,
            'predicted_price': predicted_price,
            'expected_return': expected_return,
            'signal':          '🟢 BUY' if expected_return > 1 else
                               '🔴 SELL' if expected_return < -1 else '⚪ HOLD'
        }
    except Exception as exc:
        return {'ticker': ticker, 'error': str(exc)}


print('Running quick forecasts for all holdings (this may take ~1 min) …')
forecast_results = [quick_forecast(t) for t in tickers_list]
forecast_df = pd.DataFrame(forecast_results)

display(HTML('<h3 style="color:#58a6ff">📊 5-Day Forward Forecast — All Holdings</h3>'))
display(forecast_df.style
        .format({'current_price': '${:.2f}', 'predicted_price': '${:.2f}',
                 'expected_return': '{:+.2f}%'}, na_rep='N/A')
        .map(lambda v: 'color:#3fb950' if isinstance(v, str) and 'BUY' in v
                  else 'color:#f85149' if isinstance(v, str) and 'SELL' in v else '',
                  subset=['signal'])
        .set_table_styles([{'selector': 'th', 'props': [('color', '#58a6ff')]}]))

---
## ⚙️ Section 6 — GPU / CPU Performance Info

In [ ]:
# ── 20. Detailed GPU / CPU info ───────────────────────────────────────────────
import platform, multiprocessing

cpu_count  = multiprocessing.cpu_count()
python_ver = sys.version.split()[0]
os_info    = platform.platform()

gpu_html = ''
try:
    import torch
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        gpu_html = (
            f"<tr><td><b>GPU</b></td><td style='color:#00ff88'>"
            f"{props.name}</td></tr>"
            f"<tr><td><b>VRAM</b></td><td style='color:#00ff88'>"
            f"{props.total_memory/1e9:.1f} GB</td></tr>"
            f"<tr><td><b>CUDA Compute</b></td>"
            f"<td style='color:#00ff88'>{props.major}.{props.minor}</td></tr>"
            f"<tr><td><b>PyTorch CUDA</b></td>"
            f"<td style='color:#00ff88'>{torch.version.cuda}</td></tr>"
            f"<tr><td><b>GPU Usage</b></td>"
            f"<td style='color:#00ff88'>✅ ACTIVE for tensor ops</td></tr>"
        )
    else:
        gpu_html = (
            "<tr><td><b>GPU</b></td><td style='color:#ffaa00'>"
            "No CUDA GPU detected — CPU only</td></tr>"
        )
except ImportError:
    gpu_html = (
        "<tr><td><b>GPU</b></td><td style='color:#ff6666'>"
        "PyTorch not installed — install with <code>pip install torch</code></td></tr>"
    )

display(HTML(
    f"<div style='background:#0d1117;color:#c9d1d9;padding:20px;border-radius:10px;"
    f"border:1px solid #30363d;width:100%;box-sizing:border-box;'>"
    f"<h3 style='color:#58a6ff'>⚙️ System Information</h3>"
    f"<table style='width:60%;border-collapse:collapse;font-family:monospace;font-size:13px;'>"
    f"<tr><td><b>OS</b></td><td>{os_info}</td></tr>"
    f"<tr><td><b>Python</b></td><td>{python_ver}</td></tr>"
    f"<tr><td><b>CPU Cores</b></td><td>{cpu_count}</td></tr>"
    f"{gpu_html}"
    f"</table>"
    f"<br><p style='color:#8b949e;font-size:12px;'>"
    f"💡 <b>To use the GPU</b>: install <code>torch</code> with CUDA support from "
    f"<a style='color:#58a6ff' href='https://pytorch.org/get-started/locally/' "
    f"target='_blank'>pytorch.org/get-started/locally</a>. "
    f"sklearn / Prophet models always run on CPU via multi-threaded NumPy (n_jobs=-1)."
    f"</p></div>"
))